# Sensitive Data Blast Radius (BigID × Sentinel Data Lake)

**Question:** If a single identity is compromised, how many sensitive BigID assets are at risk?

Builds a graph of `User → Asset` edges where the user has Read/Write/FullControl over PHI / GDPR / Confidential / Restricted classified data, then computes per-user blast radius.


In [ ]:
# === Setup: connect to the Microsoft Sentinel data lake ===
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

data_provider = MicrosoftSentinelProvider(spark)

WORKSPACE = "<YOUR_SENTINEL_WORKSPACE_NAME>"
TABLE = "BigIDDSPMCatalog_CL"

# Pull last 30 days of BigID catalog rows
df = data_provider.read_table(TABLE, WORKSPACE)
df = df.filter(F.col("TimeGenerated") > F.expr("current_timestamp() - INTERVAL 30 DAYS"))
df.printSchema()
print("Row count:", df.count())


## Build the User → Asset edges

In [ ]:
# Explode AssetPermissions (JSON) into flat (User, Asset, Classification) rows.
from pyspark.sql.functions import from_json, expr, col, explode, lit
from pyspark.sql.types import MapType, ArrayType

perm_schema = "Read array<string>, Write array<string>, FullControl array<string>"

sens = (
    df
    .withColumn("perms", from_json(col("AssetPermissions"), perm_schema))
    .filter(
        (col("Classification").contains("PHI")) |
        (col("Classification").contains("GDPR")) |
        (col("Classification").contains("Restricted")) |
        (col("Classification").contains("Confidential"))
    )
    .select(
        "AssetID", "AssetSource", "Classification",
        F.array_union(
            F.coalesce(col("perms.Read"), F.array()),
            F.array_union(
                F.coalesce(col("perms.Write"), F.array()),
                F.coalesce(col("perms.FullControl"), F.array()),
            ),
        ).alias("Users"),
    )
)

edges = sens.withColumn("User", explode(col("Users"))).select("User", "AssetID", "Classification", "AssetSource")
edges.show(20, truncate=False)


## Compute blast radius (top 25 users by sensitive assets reachable)

In [ ]:
result = (
    edges.groupBy("User")
    .agg(
        F.countDistinct("AssetID").alias("SensitiveAssetsReachable"),
        F.collect_set("Classification").alias("Classifications"),
        F.collect_set("AssetSource").alias("Sources"),
    )
    .orderBy(F.desc("SensitiveAssetsReachable"))
    .limit(25)
)
result.show(truncate=False)

# For graph viz: flatten back to User → AssetID edges, limited
result = (
    edges.join(result.select("User"), "User", "inner")
    .select("User", "AssetID")
    .limit(150)
)


## Visualize

In [ ]:
# === Visualize as a graph ===
import matplotlib.pyplot as plt
import networkx as nx

pdf = result.toPandas()
print(f"Edges to draw: {len(pdf)}")
display(pdf.head(50))

G = nx.DiGraph()
for _, row in pdf.iterrows():
    src = str(row.iloc[0])
    dst = str(row.iloc[1])
    G.add_edge(src, dst)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 0].values],
    node_color="#1f77b4", node_size=900, alpha=0.85,
)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 1].values and n not in pdf.iloc[:, 0].values],
    node_color="#d62728", node_size=900, alpha=0.85,
)
nx.draw_networkx_edges(G, pos, arrows=True, edge_color="#888", alpha=0.6, width=1.2)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Sensitive Data Blast Radius — Top Users → BigID Assets", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()
